# Lab 20: Satellite to Map — Pix2Pix on Real Data

**Goal:** Apply Pix2Pix to the Maps dataset — the original demo from the paper.

What we'll do:
1. Download and prepare the Maps dataset
2. Build full Pix2Pix (U-Net + PatchGAN)
3. Train: Satellite → Map
4. Train: Map → Satellite (reverse direction)
5. Compare both directions
6. Visualize PatchGAN attention on real images

This is the capstone lab (Unit 5 · L20) — everything from the course comes together.

**Notation:** x = input image (satellite), y = **target image** (map). Pix2Pix = L10's conditional GAN, conditioned on an image.

**Runtime (measured on an Apple-silicon laptop; two directions):** the U-Net has 54.4 M parameters. A 256×256 step at batch 4 takes ≈ 0.16 s on `mps` and ≈ 0.65 s on a laptop CPU. So the notebook picks the size: CUDA = all 1,096 pairs × 50 epochs; `mps` = all pairs × 20 epochs (≈ 30 min total); CPU = 400 pairs × 10 epochs (≈ 25 min total). Expect blurrier maps with the shorter runs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)

# Training size by device (see runtime note above)
TRAIN_PAIRS = {"cuda": None, "mps": None, "cpu": 400}[device.type]
EPOCHS      = {"cuda": 50,   "mps": 20,   "cpu": 10}[device.type]
print(f"Using: {device}")

## 1. Download the Maps Dataset

The dataset is from the Pix2Pix paper (aerial photos ↔ Google Maps tiles). Each 1200×600 JPG holds a satellite photo (left) and its matching map (right): 1,096 training and 1,098 validation pairs, ~240 MB download.

In [ ]:
import urllib.request

data_dir = Path("data/maps")

if not (data_dir / "train").exists():
    url = "https://efrosgans.eecs.berkeley.edu/pix2pix/datasets/maps.tar.gz"
    tar_path = "data/maps.tar.gz"
    os.makedirs("data", exist_ok=True)

    print("Downloading Maps dataset (~240MB)...")
    urllib.request.urlretrieve(url, tar_path)
    print("Download complete. Extracting...")

    import tarfile
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall("data")
    os.remove(tar_path)
    print("Done!")
else:
    print("Maps dataset already exists.")

train_dir = data_dir / "train"
val_dir = data_dir / "val"
print(f"Training images: {len(list(train_dir.glob('*.jpg')))}")
print(f"Validation images: {len(list(val_dir.glob('*.jpg')))}")

In [ ]:
# Look at a raw image — satellite + map side by side
sample_path = sorted(train_dir.glob("*.jpg"))[0]
sample_img = Image.open(sample_path)
print(f"Raw image size: {sample_img.size}")

plt.figure(figsize=(10, 4))
plt.imshow(sample_img)
plt.title(f"Raw paired image — left: satellite, right: map ({sample_img.size})")
plt.axis("off")
plt.show()

## 2. Dataset Class

Split each image into satellite (left half) and map (right half).
Resize to 256×256 for training.

In [ ]:
class MapsDataset(Dataset):
    """Loads paired satellite/map images.
    Each file has satellite on the left, map on the right."""

    def __init__(self, root_dir, img_size=256, augment=True):
        self.files = sorted(Path(root_dir).glob("*.jpg"))
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        w, h = img.size

        # Split: left = satellite, right = map
        # (In this dataset, left half is the input, right half is the target)
        satellite = img.crop((0, 0, w // 2, h))
        map_img   = img.crop((w // 2, 0, w, h))

        # Resize
        resize = transforms.Resize((self.img_size, self.img_size), antialias=True)
        satellite = resize(satellite)
        map_img   = resize(map_img)

        # Augmentation: random horizontal flip (same for both!)
        if self.augment and torch.rand(1).item() > 0.5:
            satellite = transforms.functional.hflip(satellite)
            map_img   = transforms.functional.hflip(map_img)

        # To tensor and normalize to [-1, 1]
        to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        satellite = to_tensor(satellite)
        map_img   = to_tensor(map_img)

        return satellite, map_img


train_dataset = MapsDataset(train_dir, augment=True)
if TRAIN_PAIRS:
    train_dataset.files = train_dataset.files[:TRAIN_PAIRS]
val_dataset   = MapsDataset(val_dir, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train: {len(train_dataset)} pairs")
print(f"Val:   {len(val_dataset)} pairs")

In [ ]:
# Visualize some pairs
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    sat, mp = train_dataset[i]
    # Undo normalization for display
    sat_vis = (sat.permute(1, 2, 0) + 1) / 2
    map_vis = (mp.permute(1, 2, 0) + 1) / 2

    axes[0, i].imshow(sat_vis.clamp(0, 1))
    axes[0, i].axis("off")
    axes[1, i].imshow(map_vis.clamp(0, 1))
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Satellite", fontsize=12, rotation=0, labelpad=50)
axes[1, 0].set_ylabel("Map", fontsize=12, rotation=0, labelpad=50)
plt.suptitle("Paired Data: Satellite ↔ Map", fontsize=14)
plt.tight_layout()
plt.show()

## 3. U-Net Generator (256×256, RGB)

Full-scale U-Net from the Pix2Pix paper: 8 encoder layers, 7 decoder blocks + a final up-convolution, skip connections at each level. Dropout 0.5 in the first three decoder blocks. (The paper trains with batch 1; we use batch 4 for speed.)

In [ ]:
class UNetDown(nn.Module):
    """Encoder block: Conv → BN → LeakyReLU."""
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetUp(nn.Module):
    """Decoder block: ConvTranspose → BN → ReLU → (Dropout)."""
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.model(x)
        return torch.cat([x, skip], dim=1)  # skip connection!


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3):
        super().__init__()
        # Encoder
        self.e1 = UNetDown(in_ch, 64, normalize=False)  # 256→128
        self.e2 = UNetDown(64, 128)                      # 128→64
        self.e3 = UNetDown(128, 256)                     # 64→32
        self.e4 = UNetDown(256, 512)                     # 32→16
        self.e5 = UNetDown(512, 512)                     # 16→8
        self.e6 = UNetDown(512, 512)                     # 8→4
        self.e7 = UNetDown(512, 512)                     # 4→2
        self.e8 = UNetDown(512, 512, normalize=False)    # 2→1

        # Decoder (in_ch doubled by skip connections)
        self.d1 = UNetUp(512, 512, dropout=True)     # 1→2
        self.d2 = UNetUp(1024, 512, dropout=True)    # 2→4
        self.d3 = UNetUp(1024, 512, dropout=True)    # 4→8
        self.d4 = UNetUp(1024, 512)                  # 8→16
        self.d5 = UNetUp(1024, 256)                  # 16→32
        self.d6 = UNetUp(512, 128)                   # 32→64
        self.d7 = UNetUp(256, 64)                    # 64→128

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_ch, 4, 2, 1), # 128→256
            nn.Tanh(),
        )

    def forward(self, x):
        # Encode
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)
        e6 = self.e6(e5)
        e7 = self.e7(e6)
        e8 = self.e8(e7)    # bottleneck (1x1)

        # Decode with skip connections
        d1 = self.d1(e8, e7)
        d2 = self.d2(d1, e6)
        d3 = self.d3(d2, e5)
        d4 = self.d4(d3, e4)
        d5 = self.d5(d4, e3)
        d6 = self.d6(d5, e2)
        d7 = self.d7(d6, e1)

        return self.final(d7)


gen = UNetGenerator().to(device)
test_in = torch.randn(1, 3, 256, 256, device=device)
print(f"U-Net: {test_in.shape} -> {gen(test_in).shape}")
print(f"Parameters: {sum(p.numel() for p in gen.parameters()):,}")

## 4. PatchGAN Discriminator (70×70)

Sees input + output concatenated (6 channels). Outputs a grid of real/fake scores.

In [ ]:
class PatchGAN(nn.Module):
    def __init__(self, in_ch=6):  # 3 (input) + 3 (output)
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Conv2d(512, 1, 4, 1, 1),  # no activation
        )

    def forward(self, input_img, output_img):
        x = torch.cat([input_img, output_img], dim=1)
        return self.model(x)


disc = PatchGAN().to(device)
a = torch.randn(1, 3, 256, 256, device=device)
b = torch.randn(1, 3, 256, 256, device=device)
print(f"PatchGAN: 2x{a.shape} -> {disc(a, b).shape}")
print(f"Parameters: {sum(p.numel() for p in disc.parameters()):,}")

## 5. Train: Satellite → Map

In [ ]:
gen  = UNetGenerator().to(device)
disc = PatchGAN().to(device)

opt_G = torch.optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))

bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()
lambda_L1 = 100

epochs = EPOCHS
d_losses, g_losses = [], []

print(f"Training Satellite → Map for {epochs} epochs...")
print(f"Batch size: {train_loader.batch_size}, Lambda L1: {lambda_L1}")
print()

In [ ]:
for epoch in range(epochs):
    d_sum, g_sum, n = 0, 0, 0

    for satellite, map_target in train_loader:
        satellite  = satellite.to(device)   # input
        map_target = map_target.to(device)   # target

        # --- D step ---
        fake_map = gen(satellite).detach()

        d_real = disc(satellite, map_target)
        d_fake = disc(satellite, fake_map)
        loss_D = (bce(d_real, torch.ones_like(d_real))
                + bce(d_fake, torch.zeros_like(d_fake))) / 2

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- G step ---
        fake_map = gen(satellite)
        d_fake = disc(satellite, fake_map)

        loss_GAN = bce(d_fake, torch.ones_like(d_fake))
        loss_L1  = l1(fake_map, map_target)
        loss_G   = loss_GAN + lambda_L1 * loss_L1

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        d_sum += loss_D.item()
        g_sum += loss_G.item()
        n += 1

    d_losses.append(d_sum / n)
    g_losses.append(g_sum / n)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | D: {d_losses[-1]:.4f} "
              f"| G: {g_losses[-1]:.4f} | L1: {loss_L1:.4f}")

print("\nTraining complete!")

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(d_losses, label="D", alpha=0.8)
ax.plot(g_losses, label="G", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Satellite → Map Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Results: Satellite → Map

In [ ]:
def show_results(gen, data_loader, title, n_show=4, direction="sat2map"):
    """Display input / generated / ground truth."""
    gen.eval()
    sat_batch, map_batch = next(iter(data_loader))
    sat_batch = sat_batch.to(device)
    map_batch = map_batch.to(device)

    if direction == "sat2map":
        input_imgs = sat_batch
        target_imgs = map_batch
        labels = ["Satellite\n(input)", "Generated\nmap", "Real\nmap"]
    else:
        input_imgs = map_batch
        target_imgs = sat_batch
        labels = ["Map\n(input)", "Generated\nsatellite", "Real\nsatellite"]

    with torch.no_grad():
        generated = gen(input_imgs).cpu()

    input_cpu = input_imgs.cpu()
    target_cpu = target_imgs.cpu()

    fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 10))
    for i in range(n_show):
        axes[0, i].imshow((input_cpu[i].permute(1,2,0) + 1)/2)
        axes[0, i].axis("off")
        axes[1, i].imshow((generated[i].permute(1,2,0).clamp(-1,1) + 1)/2)
        axes[1, i].axis("off")
        axes[2, i].imshow((target_cpu[i].permute(1,2,0) + 1)/2)
        axes[2, i].axis("off")

    for row, lbl in enumerate(labels):
        axes[row, 0].set_ylabel(lbl, fontsize=12, rotation=0, labelpad=60)

    plt.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()
    gen.train()


show_results(gen, val_loader, "Satellite → Map (Validation Set)", direction="sat2map")

## 7. Reverse Direction: Map → Satellite

Same architecture, swap input and target. This direction is HARDER —
the generator must synthesize realistic aerial textures from colored blocks.

**Predict first:** which direction will have the higher L1 error on the validation set?

In [ ]:
gen_rev  = UNetGenerator().to(device)
disc_rev = PatchGAN().to(device)

opt_G_rev = torch.optim.Adam(gen_rev.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D_rev = torch.optim.Adam(disc_rev.parameters(), lr=2e-4, betas=(0.5, 0.999))

d_losses_rev, g_losses_rev = [], []

print(f"Training Map → Satellite for {epochs} epochs...")
print()

for epoch in range(epochs):
    d_sum, g_sum, n = 0, 0, 0

    for satellite, map_img in train_loader:
        # REVERSE: map is input, satellite is target
        input_img = map_img.to(device)
        target    = satellite.to(device)

        # D step
        fake = gen_rev(input_img).detach()
        d_real, d_fake = disc_rev(input_img, target), disc_rev(input_img, fake)
        loss_D = (bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))) / 2
        opt_D_rev.zero_grad(); loss_D.backward(); opt_D_rev.step()

        # G step
        fake = gen_rev(input_img)
        d_fake = disc_rev(input_img, fake)
        loss_G = bce(d_fake, torch.ones_like(d_fake)) + lambda_L1 * l1(fake, target)
        opt_G_rev.zero_grad(); loss_G.backward(); opt_G_rev.step()

        d_sum += loss_D.item()
        g_sum += loss_G.item()
        n += 1

    d_losses_rev.append(d_sum / n)
    g_losses_rev.append(g_sum / n)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | D: {d_losses_rev[-1]:.4f} | G: {g_losses_rev[-1]:.4f}")

print("\nDone!")

In [ ]:
show_results(gen_rev, val_loader, "Map → Satellite (Validation Set)", direction="map2sat")

## 8. Compare Both Directions Side by Side

In [ ]:
gen.eval()
gen_rev.eval()

sat_batch, map_batch = next(iter(val_loader))
sat_batch = sat_batch.to(device)
map_batch = map_batch.to(device)

with torch.no_grad():
    fake_map = gen(sat_batch).cpu()       # sat → map
    fake_sat = gen_rev(map_batch).cpu()   # map → sat

n = 3
fig, axes = plt.subplots(2, 3 * n, figsize=(4 * n * 3, 8))

# Flatten for easier iteration
for i in range(n):
    col = i * 3
    # Row 0: Sat → Map
    axes[0, col].imshow((sat_batch[i].cpu().permute(1,2,0) + 1)/2)
    axes[0, col].set_title("Satellite", fontsize=9)
    axes[0, col+1].imshow((fake_map[i].permute(1,2,0).clamp(-1,1) + 1)/2)
    axes[0, col+1].set_title("Generated Map", fontsize=9)
    axes[0, col+2].imshow((map_batch[i].cpu().permute(1,2,0) + 1)/2)
    axes[0, col+2].set_title("Real Map", fontsize=9)

    # Row 1: Map → Sat
    axes[1, col].imshow((map_batch[i].cpu().permute(1,2,0) + 1)/2)
    axes[1, col].set_title("Map", fontsize=9)
    axes[1, col+1].imshow((fake_sat[i].permute(1,2,0).clamp(-1,1) + 1)/2)
    axes[1, col+1].set_title("Generated Satellite", fontsize=9)
    axes[1, col+2].imshow((sat_batch[i].cpu().permute(1,2,0) + 1)/2)
    axes[1, col+2].set_title("Real Satellite", fontsize=9)

for ax in axes.flat:
    ax.axis("off")

axes[0, 0].set_ylabel("Sat→Map", fontsize=12, rotation=0, labelpad=50)
axes[1, 0].set_ylabel("Map→Sat", fontsize=12, rotation=0, labelpad=50)

plt.suptitle("Both Directions: Satellite ↔ Map", fontsize=15)
plt.tight_layout()
plt.show()

print("Sat→Map: Should have correct road/building positions, right colors.")
print("Map→Sat: Harder — must generate realistic textures from colored blocks.")

## 9. PatchGAN Visualization on Real Data

See what patches the discriminator finds realistic vs fake.

In [ ]:
disc.eval()

with torch.no_grad():
    fake_maps = gen(sat_batch).cpu()
    d_real_scores = disc(sat_batch, map_batch).cpu()
    d_fake_scores = disc(sat_batch, gen(sat_batch)).cpu()

n = 3
fig, axes = plt.subplots(4, n, figsize=(5 * n, 14))

for i in range(n):
    axes[0, i].imshow((map_batch[i].cpu().permute(1,2,0) + 1)/2)
    axes[0, i].axis("off")
    axes[0, i].set_title("Real map")

    scores = torch.sigmoid(d_real_scores[i].squeeze())
    axes[1, i].imshow(scores, cmap="RdYlGn", vmin=0, vmax=1)
    axes[1, i].axis("off")
    axes[1, i].set_title(f"D(real): {scores.mean():.2f}")

    axes[2, i].imshow((fake_maps[i].permute(1,2,0).clamp(-1,1) + 1)/2)
    axes[2, i].axis("off")
    axes[2, i].set_title("Fake map")

    scores = torch.sigmoid(d_fake_scores[i].squeeze())
    axes[3, i].imshow(scores, cmap="RdYlGn", vmin=0, vmax=1)
    axes[3, i].axis("off")
    axes[3, i].set_title(f"D(fake): {scores.mean():.2f}")

axes[0, 0].set_ylabel("Real\nmap", fontsize=11, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("D scores\n(real)", fontsize=11, rotation=0, labelpad=40)
axes[2, 0].set_ylabel("Fake\nmap", fontsize=11, rotation=0, labelpad=40)
axes[3, 0].set_ylabel("D scores\n(fake)", fontsize=11, rotation=0, labelpad=40)

plt.suptitle("PatchGAN Scores on Real Data (green=real, red=fake)", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Course Complete!

You've just applied the complete Pix2Pix pipeline to a real-world dataset.

**What you built across the five units:**

| Unit | Labs | What you built |
|-----|------|---------------|
| Unit 1 | P1–P3, Lab 0 | A learning neuron, backprop by hand, a classifier, the same in PyTorch |
| Unit 1 | Labs 01–04 | First GAN (the number 7), MNIST GAN, DCGAN, training tricks |
| Unit 2 | Labs 08–11 | WGAN, WGAN-GP, conditional GAN, controllable generation |
| Unit 3 | Labs 13, 15 | FID from scratch, mini-StyleGAN |
| Unit 4 | Lab 17 | Pix2Pix — edges to digits (U-Net + PatchGAN) |
| Unit 5 | Lab 19 | CycleGAN — unpaired domain translation |
| Unit 5 | **Lab 20** | **Satellite → Map — real-world Pix2Pix** |

**Your turn:** compute FID (Unit 3, lab-13) between your generated maps and the real validation maps. Does the number agree with your eyes?
